# 01clean: from raw files to tidy tables

Reads every raw file in `data/raw` and writes tidy tables to `data/clean`:

| Output | What one row is |
|---|---|
| `lines.csv` | one line as it appears in the data, with an English name, the route it belongs to and its operator group |
| `stations.csv` | one station on one line, with its base name, latest full name, English name and Seoul Metro code (Lines 1 to 8) |
| `ridership_daily.parquet` | one day, line and station: boardings and alightings (2015 to 2026) |
| `hourly.parquet` | one day, station, gate type and hour: passengers (Seoul Metro, 2024) |
| `crowding.parquet` | one quarter, day type, station, direction and half hour: crowding % |
| `operations.csv` | one Seoul Metro line: length, stations, fleet, headways, trips (March 2024) |

Parquet is a compact file format that keeps column types (dates stay dates). It needs the `pyarrow` package.

Run time on the full data: about one minute.

## Section 0: settings

In [1]:
from pathlib import Path

RAW = Path("data/raw")                       # <- raw files
CLEAN = Path("data/clean")                   # <- output folder (created if missing)
ENCODINGS = ["utf-8-sig", "cp949"]           # <- encodings to try, in this order

# Real station renames (old base name -> new base name). Subtitles in brackets are removed
# automatically, so only true renames need to be listed here.
STATION_RENAMES = {                          # <- add more if a station shows up twice
    "신천": "잠실새내",          # Line 2, renamed 2016
    "당고개": "불암산",          # Line 4, renamed 2025
    "뚝섬유원지": "자양",        # Line 7, renamed 2024
    "지제": "평택지제",          # Gyeongbu Line, renamed 2021
    "쌍용동": "쌍용",            # Janghang Line, renamed 2016
    "인천국제공항": "인천공항1터미널",  # AREX, renamed 2018
    "미아삼거리": "미아사거리",  # Line 4, renamed 2010 (old name still used in some crowding files)
}

# Line names that changed over time (old -> new)
LINE_RENAMES = {"9호선2단계": "9호선2~3단계"}  # <-

# When the same day is in a downloaded file and in an API file, keep the downloaded file
PREFER_MANUAL_OVER_API = True                # <-

# Crowding: 100% means about this many people in one car (Seoul Metro definition: seats full = 34%)
PEOPLE_AT_100_PCT = 160                      # <-

# English station names: official names from Seoul Metro's multilingual list; other stations are romanised automatically
STATION_EN_OVERRIDES = {"인천공항1터미널": "Incheon Airport Terminal 1", "인천공항2터미널": "Incheon Airport Terminal 2"}

## Section 1: tools

In [2]:
import csv
import io
import re
import time

import pandas as pd

CLEAN.mkdir(parents=True, exist_ok=True)
T0 = time.time()


def decode(path):
    """Return the text of a file, trying each encoding in ENCODINGS."""
    raw = path.read_bytes()
    for enc in ENCODINGS:
        try:
            return raw.decode(enc).lstrip("\ufeff"), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No encoding in {ENCODINGS} worked for {path.name}")


def read_csv_any(path, **kwargs):
    text, enc = decode(path)
    return pd.read_csv(io.StringIO(text), **kwargs)


def base_name(name):
    """Station name without the subtitle in brackets and without spaces, after known renames.
    Example: '잠실(송파구청)' -> '잠실', '동두천 중앙' -> '동두천중앙', '신천' -> '잠실새내'."""
    b = re.sub(r"\(.*\)$", "", str(name).strip()).replace(" ", "")
    return STATION_RENAMES.get(b, b)


def to_number(s):
    return pd.to_numeric(s.astype(str).str.strip().replace({"": None, "nan": None}), errors="coerce")


def done(msg):
    print(f"[{time.time() - T0:5.1f}s] {msg}")

## Section 2: lines lookup

One row per line name used in the ridership files. In the card data, the long KORAIL routes are split into their historic sections
(for example Line 1 north of Seoul Station is 경원선, south is 경부선), so each section gets the `route` it belongs to.
`1호선` to `8호선` are only the Seoul Metro parts of each route.

In [3]:
LINES = [
    # line_ko,        line_en,                          route,               operator_group
    ("1호선",         "Line 1 (Seoul Metro section)",   "Line 1",            "Seoul Metro"),
    ("2호선",         "Line 2",                         "Line 2",            "Seoul Metro"),
    ("3호선",         "Line 3 (Seoul Metro section)",   "Line 3",            "Seoul Metro"),
    ("4호선",         "Line 4 (Seoul Metro section)",   "Line 4",            "Seoul Metro"),
    ("5호선",         "Line 5",                         "Line 5",            "Seoul Metro"),
    ("6호선",         "Line 6",                         "Line 6",            "Seoul Metro"),
    ("7호선",         "Line 7",                         "Line 7",            "Seoul Metro"),
    ("8호선",         "Line 8",                         "Line 8",            "Seoul Metro"),
    ("9호선",         "Line 9 (phase 1)",               "Line 9",            "Other operators"),
    ("9호선2~3단계",  "Line 9 (phases 2 and 3)",        "Line 9",            "Seoul Metro"),
    ("경부선",        "Gyeongbu Line (Line 1)",         "Line 1",            "KORAIL"),
    ("경원선",        "Gyeongwon Line (Line 1)",        "Line 1",            "KORAIL"),
    ("경인선",        "Gyeongin Line (Line 1)",         "Line 1",            "KORAIL"),
    ("장항선",        "Janghang Line (Line 1)",         "Line 1",            "KORAIL"),
    ("일산선",        "Ilsan Line (Line 3)",            "Line 3",            "KORAIL"),
    ("과천선",        "Gwacheon Line (Line 4)",         "Line 4",            "KORAIL"),
    ("안산선",        "Ansan Line (Line 4)",            "Line 4",            "KORAIL"),
    ("분당선",        "Bundang Line",                   "Suin-Bundang",      "KORAIL"),
    ("수인선",        "Suin Line",                      "Suin-Bundang",      "KORAIL"),
    ("경의선",        "Gyeongui Line",                  "Gyeongui-Jungang",  "KORAIL"),
    ("중앙선",        "Jungang Line",                   "Gyeongui-Jungang",  "KORAIL"),
    ("경춘선",        "Gyeongchun Line",                "Gyeongchun",        "KORAIL"),
    ("경강선",        "Gyeonggang Line",                "Gyeonggang",        "KORAIL"),
    ("공항철도 1호선", "AREX (Airport Railroad)",        "AREX",              "Other operators"),
    ("서해선",        "Seohae Line",                    "Seohae",            "Other operators"),
    ("신림선",        "Sillim Line",                    "Sillim",            "Other operators"),
    ("우이신설선",    "Ui-Sinseol Line",                "Ui-Sinseol",        "Other operators"),
]
lines = pd.DataFrame(LINES, columns=["line_ko", "line_en", "route", "operator_group"])
lines["line_order"] = range(1, len(lines) + 1)
lines

,line_ko,line_en,route,operator_group,line_order
0,1호선,Line 1 (Seoul Metro section),Line 1,Seoul Metro,1
1,2호선,Line 2,Line 2,Seoul Metro,2
2,3호선,Line 3 (Seoul Metro section),Line 3,Seoul Metro,3
3,4호선,Line 4 (Seoul Metro section),Line 4,Seoul Metro,4
4,5호선,Line 5,Line 5,Seoul Metro,5
5,6호선,Line 6,Line 6,Seoul Metro,6
6,7호선,Line 7,Line 7,Seoul Metro,7
7,8호선,Line 8,Line 8,Seoul Metro,8
8,9호선,Line 9 (phase 1),Line 9,Other operators,9
9,9호선2~3단계,Line 9 (phases 2 and 3),Line 9,Seoul Metro,10


## Section 3: daily ridership (2015 to 2026)

Reads the yearly and monthly downloaded files plus the API files, repairs glued rows, keeps the first six fields,
removes the API copy of days that are also in a downloaded file, and adds the base station name.

In [4]:
RID_COLS = ["date", "line_ko", "station_raw", "board", "alight", "registered"]
GLUED = re.compile(r'(\d")("\d{8}",")')   # end of one row stuck to the start of the next


def read_ridership(path):
    text, enc = decode(path)
    text, n_fixed = GLUED.subn(r"\1\n\2", text)
    rows = [r[:6] for r in csv.reader(io.StringIO(text))][1:]
    df = pd.DataFrame([r for r in rows if r], columns=RID_COLS)
    df["source"] = "api" if path.parent.name == "api" else "download"
    df["file"] = path.name
    return df, n_fixed


files = sorted((RAW / "ridership").glob("*.csv")) + sorted((RAW / "api").glob("*.csv"))
parts, fixed = [], 0
for f in files:
    df, n = read_ridership(f)
    parts.append(df)
    fixed += n
rid = pd.concat(parts, ignore_index=True)
done(f"read {len(files)} files, {len(rid):,} rows, repaired {fixed} glued row(s)")

rid["date"] = pd.to_datetime(rid["date"], format="%Y%m%d")
rid["board"] = to_number(rid["board"])
rid["alight"] = to_number(rid["alight"])
rid["line_ko"] = rid["line_ko"].replace(LINE_RENAMES)
rid["station_ko"] = rid["station_raw"].map(base_name)

if PREFER_MANUAL_OVER_API:
    manual_days = set(rid.loc[rid["source"] == "download", "date"])
    overlap = (rid["source"] == "api") & rid["date"].isin(manual_days)
    rid = rid[~overlap]
    done(f"dropped {overlap.sum():,} API rows for days already in downloaded files")

unknown = sorted(set(rid["line_ko"]) - set(lines["line_ko"]))
assert not unknown, f"lines missing from the LINES table in Section 2: {unknown}"
print(f"missing numbers: {rid[['board', 'alight']].isna().sum().to_dict()}")

[ 10.5s] read 54 files, 2,555,538 rows, repaired 1 glued row(s)
[ 19.1s] dropped 18,489 API rows for days already in downloaded files
missing numbers: {'board': 0, 'alight': 0}


In [5]:
# Sum rows that share a day, line and base station name
# (the small duplicate rows at a few Gyeongui Line stations, and any renamed station that appears twice)
multi = rid.duplicated(["date", "line_ko", "station_ko"], keep=False)
print(f"rows that share a day, line and station: {multi.sum():,}")
print(rid.loc[multi, "station_raw"].value_counts().head(8).to_string())

ridership = (rid.groupby(["date", "line_ko", "station_ko"], as_index=False)[["board", "alight"]].sum())
ridership[["board", "alight"]] = ridership[["board", "alight"]].astype("int64")
done(f"ridership: {len(ridership):,} rows, {ridership['date'].min():%Y-%m-%d} to {ridership['date'].max():%Y-%m-%d}")

# every day present?
all_days = pd.date_range(ridership["date"].min(), ridership["date"].max(), freq="D")
missing_days = all_days.difference(ridership["date"].unique())
print(f"missing days: {len(missing_days)}")

rows that share a day, line and station: 740
station_raw
서울역         592
홍대입구         54
공덕           50
디지털미디어시티     44
[ 23.4s] ridership: 2,536,679 rows, 2015-01-01 to 2026-09-30
missing days: 0


## Section 4: hourly boardings (Seoul Metro, 2024)

The 20 hour columns become one `hour` column. `hour` is the start of the hour; `5` means "before 06:00" and `24` means "after midnight".

In [ ]:
f = next((RAW / "hourly").glob("*.csv"))
hr = read_csv_any(f)
hour_cols = [c for c in hr.columns if c.endswith("시간대") or c in ("06시이전", "24시이후")]
hour_of = {c: (5 if c == "06시이전" else 24 if c == "24시이후" else int(c[:2])) for c in hour_cols}

hr = hr.rename(columns={"수송일자": "date", "호선": "line_ko", "역번호": "station_code", "역명": "station_full", "승하차구분": "gate"})
hr["date"] = pd.to_datetime(hr["date"])
hr["gate"] = hr["gate"].map({"승차": "board", "하차": "alight"})
hr["station_ko"] = hr["station_full"].map(base_name)
hourly = hr.melt(id_vars=["date", "line_ko", "station_code", "station_ko", "gate"], value_vars=hour_cols, var_name="hour", value_name="passengers")
hourly["hour"] = hourly["hour"].map(hour_of).astype("int8")
hourly["passengers"] = hourly["passengers"].astype("int32")
hourly = hourly.sort_values(["date", "line_ko", "station_code", "gate", "hour"], ignore_index=True)
done(f"hourly: {len(hourly):,} rows ({hourly['date'].nunique()} days, {hourly['station_code'].nunique()} station codes, hours {hourly['hour'].min()} to {hourly['hour'].max()})")

# check: hourly totals per day and station equal the daily card data for the same day?
h_day = (hourly[hourly["gate"] == "board"].groupby(["date", "line_ko", "station_ko"])["passengers"].sum())
r_day = ridership.set_index(["date", "line_ko", "station_ko"])["board"]
cmp = pd.concat([h_day.rename("hourly"), r_day.rename("daily")], axis=1, join="inner")
ratio = cmp["hourly"].sum() / cmp["daily"].sum()
print(f"hourly vs daily boardings on matching rows ({len(cmp):,}): hourly total is {ratio:.1%} of daily total")
# Most of this difference comes from April 2024, whose 13:00 slot is about ten times too high in the source file (02dashdata leaves April out).

[ 27.4s] hourly: 3,988,480 rows (366 days, 274 station codes, hours 5 to 24)
hourly vs daily boardings on matching rows (99,703): hourly total is 103.8% of daily total


## Section 5: crowding (10 quarters, 2023 Q4 to 2026 Q2)

Three layouts are read into one long table. The quarter comes from the date in the file name
(the 2025 Q4 file is dated 30 November). Half-hour slots are stored as their start time, from `05:30` to `00:30`.

In [7]:
SLOTS = [f"{h:02d}:{m:02d}" for h in range(5, 25) for m in (0, 30)][1:40]   # 05:30 ... 24:30
SLOTS = [s.replace("24:", "00:") for s in SLOTS]                                 # 24:00 -> 00:00
DAY_TYPES = {"평일": "weekday", "토요일": "saturday", "일요일": "sunday"}
DIRECTIONS = {"상선": "up", "하선": "down", "내선": "inner", "외선": "outer"}


def tidy_crowding(df, quarter):
    """Any of the three layouts -> long table. The 39 slot columns are always the last 39 columns."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={"요일구분": "day_type", "구분": "day_type", "호선": "line_ko", "역번호": "station_code",
                            "출발역": "station_full", "역명": "station_full", "상하구분": "direction"})
    slot_cols = list(df.columns[-39:])
    df = df.rename(columns=dict(zip(slot_cols, SLOTS)))
    df["line_ko"] = df["line_ko"].astype(str).str.replace("호선", "", regex=False).str.strip() + "호선"
    df["station_code"] = pd.to_numeric(df["station_code"], errors="coerce").astype("Int64")
    long = df.melt(id_vars=["day_type", "line_ko", "station_code", "station_full", "direction"], value_vars=SLOTS, var_name="slot", value_name="crowding_pct")
    long["crowding_pct"] = to_number(long["crowding_pct"])
    long["quarter"] = quarter
    return long


parts = []
for f in sorted((RAW / "crowding").iterdir()):
    stamp = pd.Timestamp(re.search(r"(\d{8})", f.name).group(1))
    quarter = str(stamp.to_period("Q"))            # e.g. '2024Q1'
    if f.suffix == ".csv":
        tables = [read_csv_any(f)]
    else:
        tables = [t for t in pd.read_excel(f, sheet_name=None).values() if len(t)]   # skip empty sheets
    for t in tables:
        parts.append(tidy_crowding(t, quarter))
    print(f"  {f.name}: {quarter}, {sum(len(t) for t in tables)} rows in {len(tables)} table(s)")

crowding = pd.concat(parts, ignore_index=True)
crowding["day_type"] = crowding["day_type"].astype(str).str.strip().map(DAY_TYPES)       # some files pad with spaces
crowding["direction"] = crowding["direction"].astype(str).str.strip().map(DIRECTIONS)
crowding["station_ko"] = crowding["station_full"].map(base_name)

# Where the station code is known from the hourly file, take the line and name from there
# (fixes a few rows with the wrong line or an old station name)
code_book = hourly[["station_code", "line_ko", "station_ko"]].drop_duplicates("station_code").set_index("station_code")
known = crowding["station_code"].isin(code_book.index)
fixed_rows = (known & ((crowding["line_ko"] != crowding["station_code"].map(code_book["line_ko"])) |
                       (crowding["station_ko"] != crowding["station_code"].map(code_book["station_ko"])))).sum()
crowding.loc[known, "line_ko"] = crowding.loc[known, "station_code"].map(code_book["line_ko"])
crowding.loc[known, "station_ko"] = crowding.loc[known, "station_code"].map(code_book["station_ko"])
print(f"rows whose line or station name was corrected from the hourly code list: {fixed_rows:,}")

# Codes 9001 to 9006 are extra entries at branch points (for example 성수E, 응암S). Keep them, but flag them.
crowding["branch_entry"] = crowding["station_code"] >= 9000
crowding["people_per_car"] = (crowding["crowding_pct"] / 100 * PEOPLE_AT_100_PCT).round(0)
crowding = crowding[["quarter", "day_type", "line_ko", "station_code", "station_ko", "branch_entry", "direction", "slot", "crowding_pct", "people_per_car"]]
print(f"missing values: day_type {crowding['day_type'].isna().sum()}, direction {crowding['direction'].isna().sum()}, "
      f"station_code {crowding['station_code'].isna().sum()}, crowding_pct {crowding['crowding_pct'].isna().sum()}")
print("missing crowding values by slot (mostly the last trains after midnight):")
print(crowding[crowding["crowding_pct"].isna()]["slot"].value_counts().head(4).to_string())
done(f"crowding: {len(crowding):,} rows, quarters {sorted(crowding['quarter'].unique())}")

  서울교통공사_지하철혼잡도정보_20231231.csv: 2023Q4, 1668 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20240331.csv: 2024Q1, 1668 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20240630.csv: 2024Q2, 1668 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20241231.csv: 2024Q4, 1662 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20250331.xlsx: 2025Q1, 1662 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20250630.csv: 2025Q2, 1671 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20250930.csv: 2025Q3, 1671 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20251130.csv: 2025Q4, 1671 rows in 1 table(s)
  서울교통공사_지하철혼잡도정보_20260331.xlsx: 2026Q1, 1671 rows in 3 table(s)
  서울교통공사_지하철혼잡도정보_20260630.xlsx: 2026Q2, 1671 rows in 3 table(s)
rows whose line or station name was corrected from the hourly code list: 1,326
missing values: day_type 0, direction 0, station_code 0, crowding_pct 5781
missing crowding values by slot (mostly the last trains after midnight):
slot
00:30    4505
00:00    1129
05:30     117
23:30      30
[ 31.9s] crowding: 650,637 rows, quarters ['2023Q4', '2024Q1', '2024Q

In [8]:
# rows per quarter (station and direction pairs per day type) and weekday peak per quarter
wk = crowding[crowding["day_type"] == "weekday"]
q = wk.groupby("quarter").agg(station_direction_pairs=("crowding_pct", lambda s: len(s) // len(SLOTS)), weekday_median=("crowding_pct", "median"), weekday_max=("crowding_pct", "max"))
q.round(1)

,station_direction_pairs,weekday_median,weekday_max
quarter,,,
2023Q4,556,27.7,166.2
2024Q1,556,28.2,157.0
2024Q2,556,27.5,148.3
2024Q4,554,27.9,154.8
2025Q1,554,28.2,151.9
2025Q2,557,27.4,145.7
2025Q3,557,28.4,151.5
2025Q4,557,28.2,159.4
2026Q1,557,28.8,147.7


## Section 6: train operations by line (March 2024)

The source is a small table with measures as rows. Cells with extra detail keep only the main number
(the Line 2 branch lines and the Seoul Metro and KORAIL split are noted in `note`).

In [9]:
f = next((RAW / "operations").glob("*.csv"))
ops_raw = read_csv_any(f, dtype=str).set_index("구분")
ops_raw.index = ops_raw.index.str.replace(" ", "")

MEASURES = {   # source row -> output column
    "구간": "section", "영업거리(km)": "length_km", "역수(개)": "stations", "소요시간(분)": "end_to_end_min",
    "보유차량수(칸)": "cars_owned", "열차편성수(1편성당칸수)": "trainsets", "표정속도(km/h)": "average_speed_kmh",
    "열차주행거리(평일km/일)": "train_km_weekday", "운행시격_RH(출근)": "headway_peak_min",
    "운행시격_NH(평시)": "headway_offpeak_min", "운행횟수_평일": "trips_weekday", "운행횟수_휴일": "trips_holiday",
}


def main_number(cell):
    """'48.8[5.4/6.0]' -> 48.8, '588(S87/K501)' -> 588, '16(10)' -> 16."""
    m = re.match(r"\s*([0-9.]+)", str(cell))
    return float(m.group(1)) if m else None


ops = ops_raw.loc[list(MEASURES)].T.rename(columns=MEASURES)
ops.index.name = "line_ko"
ops["cars_per_trainset"] = ops["trainsets"].str.extract(r"\((\d+)").astype(float)
for c in ops.columns:
    if c not in ("section", "cars_per_trainset"):
        ops[c] = ops[c].map(main_number)
ops.columns.name = None
ops["note"] = ""
ops.loc["1호선", "note"] = "Seoul Metro section only (Seoul Station to Cheongnyangni); trips include KORAIL through trains"
ops.loc["2호선", "note"] = "main loop; Seongsu and Sinjeong branches excluded"
ops.loc[["3호선", "4호선"], "note"] = "trips include KORAIL through trains"
ops = ops.reset_index()
ops

,line_ko,section,length_km,stations,end_to_end_min,cars_owned,trainsets,average_speed_kmh,train_km_weekday,headway_peak_min,headway_offpeak_min,trips_weekday,trips_holiday,cars_per_trainset,note
0,1호선,서울~청량리,7.8,10.0,18.0,160.0,16.0,26.0,5064.8,3.0,5.0,588.0,535.0,10.0,Seoul Metro section only (Seoul Station to Che...
1,2호선,성수~성수[성수지선/신정지선],48.8,43.0,90.0,750.0,84.0,32.5,23728.4,2.5,5.5,528.0,422.0,10.0,main loop; Seongsu and Sinjeong branches excluded
2,3호선,지축~오금,38.2,34.0,67.5,490.0,49.0,34.0,14236.4,3.0,6.5,407.0,369.0,10.0,trips include KORAIL through trains
3,4호선,진접~남태령,45.4,29.0,68.0,520.0,52.0,40.1,16019.6,2.5,5.5,480.0,408.0,10.0,trips include KORAIL through trains
4,5호선,방화~하남검단산/마천,59.8,56.0,108.5,696.0,87.0,33.1,20580.7,2.5,6.5,428.0,342.0,8.0,
5,6호선,응암순환~신내,36.3,39.0,75.0,312.0,39.0,29.0,10724.4,4.0,8.0,340.0,251.0,8.0,
6,7호선,장암~온수,46.9,42.0,87.0,616.0,77.0,32.3,18546.5,2.5,6.0,404.0,328.0,8.0,
7,8호선,암사~모란,17.7,18.0,32.5,120.0,20.0,32.7,5193.8,4.5,8.5,292.0,242.0,6.0,


## Section 7: stations lookup

One row per line and base station name found in any dataset. For Seoul Metro Lines 1 to 8 it also holds the station code
used in the hourly and crowding files.

In [10]:
latest_full = (rid.sort_values("date").groupby(["line_ko", "station_ko"])["station_raw"].last().rename("station_full"))
span = ridership.groupby(["line_ko", "station_ko"])["date"].agg(first_day="min", last_day="max")
stations = span.join(latest_full).reset_index()

codes = pd.concat([hourly[["line_ko", "station_ko", "station_code"]], crowding.loc[~crowding["branch_entry"], ["line_ko", "station_ko", "station_code"]]]).drop_duplicates()
clash = codes.groupby(["line_ko", "station_ko"])["station_code"].nunique()
print(f"line and station pairs with more than one code: {(clash > 1).sum()}")
codes = codes.drop_duplicates(["line_ko", "station_ko"])

stations = stations.merge(codes, on=["line_ko", "station_ko"], how="outer")
stations = stations.merge(lines[["line_ko", "line_order"]], on="line_ko", how="left")
stations = stations.sort_values(["line_order", "station_code", "station_ko"]).drop(columns="line_order")
stations["station_code"] = stations["station_code"].astype("Int64")

only_codes = stations[stations["first_day"].isna()]
print(f"stations: {len(stations)}, with a Seoul Metro code: {stations['station_code'].notna().sum()}")
print(f"in hourly or crowding but not in daily ridership: {len(only_codes)}")
only_codes

line and station pairs with more than one code: 0
stations: 633, with a Seoul Metro code: 278
in hourly or crowding but not in daily ridership: 1


,line_ko,station_ko,first_day,last_day,station_full,station_code
16,2호선,까치산,NaT,NaT,NaN,260


### English station names

Official English names come from Seoul Metro's multilingual station list (`data/raw/stationnames`). They are matched on the base Korean name, so a station shared with a KORAIL line (for example 서울역) gets the official name on every line.
Stations not in that list are romanised automatically (Revised Romanisation, package `korean-romanizer`). Automatic names can miss sound changes (for example 신림 becomes "Sinrim" instead of "Sillim"), so they are marked `automatic` in `station_en_source`. Fix any of them in `STATION_EN_OVERRIDES` in Section 0.

In [11]:
from korean_romanizer.romanizer import Romanizer

names_file = next((RAW / "stationnames").glob("*.csv"))
nm = read_csv_any(names_file, dtype=str)
nm["station_ko"] = nm["한글"].map(base_name)
nm["station_en"] = nm["영문"].str.replace(r"\s*\(.*\)$", "", regex=True).str.strip()   # drop the English subtitle in brackets
official = nm.dropna(subset=["station_en"]).drop_duplicates("station_ko").set_index("station_ko")["station_en"]
print(f"official English names: {len(official)} from {names_file.name}")


def auto_en(ko):
    """Automatic Revised Romanisation, first letter in capitals."""
    r = Romanizer(ko).romanize()
    return r[:1].upper() + r[1:]


stations["station_en"] = stations["station_ko"].map(official)
stations["station_en_source"] = stations["station_en"].notna().map({True: "official", False: "automatic"})
stations.loc[stations["station_en"].isna(), "station_en"] = stations.loc[stations["station_en"].isna(), "station_ko"].map(auto_en)
for ko, en in STATION_EN_OVERRIDES.items():
    stations.loc[stations["station_ko"] == ko, ["station_en", "station_en_source"]] = [en, "override"]
stations["station_label"] = stations["station_en"] + " (" + stations["station_ko"] + ")"
print(stations["station_en_source"].value_counts().to_string())
stations.loc[stations["station_en_source"] == "automatic", ["line_ko", "station_ko", "station_en"]].head(15)

official English names: 251 from 서울교통공사_역명다국어표기_20260903.csv
station_en_source
official     336
automatic    295
override       2


,line_ko,station_ko,station_en
224,7호선,까치울,Kkachiul
223,7호선,굴포천,Gulpocheon
237,7호선,부천시청,Bucheonsicheong
238,7호선,부천종합운동장,Bucheonjonghabundongjang
239,7호선,부평구청,Bupyeonggucheong
241,7호선,삼산체육관,Samsancheyukgwan
243,7호선,상동,Sangdong
248,7호선,신중동,Sinjungdong
263,7호선,춘의,Chunui
286,9호선,가양,Gayang


## Section 8: save

In [12]:
lines.to_csv(CLEAN / "lines.csv", index=False, encoding="utf-8-sig")
stations.to_csv(CLEAN / "stations.csv", index=False, encoding="utf-8-sig")
ops.to_csv(CLEAN / "operations.csv", index=False, encoding="utf-8-sig")

for name, df in [("ridership_daily", ridership), ("hourly", hourly), ("crowding", crowding)]:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype).startswith("str"):
            df[c] = df[c].astype("category")
    df.to_parquet(CLEAN / f"{name}.parquet", index=False)

for f in sorted(CLEAN.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")
done("saved")

  crowding.parquet               2.36 MB
  hourly.parquet                 7.54 MB
  lines.csv                      0.00 MB
  operations.csv                 0.00 MB
  ridership_daily.parquet       12.46 MB
  stations.csv                   0.06 MB
[ 35.8s] saved


## Section 9: checks

Quick sanity checks on the saved tables.

In [13]:
chk = pd.read_parquet(CLEAN / "ridership_daily.parquet")
year = chk.groupby(chk["date"].dt.year).agg(days=("date", "nunique"), stations=("station_ko", "size"), boardings=("board", "sum"))
year["stations"] = (year["stations"] / year["days"]).round(0)
year["boardings_per_day_million"] = (year["boardings"] / year["days"] / 1e6).round(2)
year.drop(columns="boardings").rename(columns={"stations": "station_rows_per_day"})

,days,station_rows_per_day,boardings_per_day_million
date,,,
2015,365,546.0,7.27
2016,366,556.0,7.26
2017,365,571.0,7.25
2018,365,583.0,7.30
2019,365,591.0,7.44
2020,366,593.0,5.38
2021,365,601.0,5.50
2022,365,602.0,6.14
2023,365,609.0,6.72


___
**END**